# PCA Dislocation on MMSS Swap Spreads

Applies the PCA dislocation framework (JPM May 2025 / Dallas Fed) directly to maturity-matched swap spread **changes**:

1. Run rolling PCA on daily MMSS changes across the benchmark tenor panel (2Y–30Y)
2. Reconstruct each tenor's 'fair' spread change from the top-k PCs
3. Cumulate the out-of-sample residuals (the idiosyncratic component)
4. Model cumulative residuals as an OU process → z-score
5. Trade individual tenor dislocations (outright spread) or pair/fly combinations

Uses the existing `_rolling_pca_surface` engine from `irswap_pca_rv_scanner.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime, sys, os, time, itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import QuantLib as ql

plt.style.use('ggplot')
pylab.rcParams.update({
    'legend.fontsize': 'x-large', 'figure.figsize': (18, 8),
    'axes.labelsize': 'x-large', 'axes.titlesize': 'x-large',
    'xtick.labelsize': 'large', 'ytick.labelsize': 'large',
})

sys.path.append('../../')

from BT.data_handler import TimeGrid
from BT.misc import ql_cal_date_range
from BT.query_actions import AddQueryAction, UnwindPositionsAction
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy
from BT.triggers import DateTrigger, DateTriggerRequirements
from BT.query_tearsheet import QueryBacktestTearSheet

from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.FixedRateBonds.FixedRateBondQuery import FixedRateBondQuery
from Query.FixedRateBonds.FixedRateBondValue import FixedRateBondValue
from Query.FixedRateBonds.carry_roll import load_us_treasury_gc_fixing_pct
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapValue import IRSwapValue

from BT.signals.tfp_swap_spread import (
    BENCHMARK_TENORS, REGRESSION_TENORS, CT_MAP, build_tfp_history,
)
from BT.signals.irswap_pca_rv_scanner import IRSwapPCARVConfig, _rolling_pca_surface
from BT.signals.tfp_packages import _ou_params
from RVUtils.plt_timeseries import make_secondary_axis_plot

## Config

In [ ]:
cfg = dict(
    signal_start    = datetime.date(2020, 6, 1),
    bt_start        = datetime.date(2021, 6, 1),
    bt_end          = datetime.date(2026, 5, 14),

    # PCA
    pca_window      = 520,       # ~2yr rolling PCA
    pca_input       = 'changes', # PCA on daily MMSS changes (not levels)
    n_components    = 3,         # top 3 PCs (level/slope/curvature)

    # Z-score on cumulative residuals
    zscore_window   = 260,       # ~1yr decoupled lookback for z-score
    z_entry         = 1.5,       # entry threshold
    z_exit          = 0.25,      # exit near zero (mean-reversion target)
    max_hold        = 60,        # max holding period

    # Trade universe
    tenors          = ['2Y', '3Y', '5Y', '7Y', '10Y', '30Y'],
    trade_mode      = 'outright',  # 'outright', 'curve', 'fly'

    # Risk
    risk_bpv        = 100_000,
    unwind_fee_bps  = 0.5,
    specialness_bps = 10.0,

    # Data
    irs_source      = 'ERIS_EOD_LIVE-RL_BASIC',
    frb_source      = 'USTS_FEDINVEST_WSJ_LIVE-QL',
    cache_path      = os.path.abspath(os.path.join(
        os.getcwd(), '..', '..', 'BT', 'results', 'tfp_screener', 'tfp_history.parquet'
    )),
)

for k, v in cfg.items():
    print(f'  {k}: {v}')

## Load MMSS Panel

In [ ]:
curve_mdp = IRSwapsMDP(source=cfg['irs_source'])
usts_mdp = FixedRateBondsMDP(source='USTS_FEDINVEST_WSJ_LIVE-RL')

history = build_tfp_history(
    start_date=cfg['signal_start'], end_date=cfg['bt_end'],
    curve_mdp=curve_mdp, usts_mdp=usts_mdp,
    cache_path=cfg['cache_path'], show_progress=True,
)

# Build MMSS panel: (date x tenor) from the cached TFP history
mmss_panel = pd.DataFrame({
    t: history[f'mmss_{t}'] for t in cfg['tenors'] if f'mmss_{t}' in history.columns
}).dropna()

print(f'MMSS panel: {mmss_panel.shape[0]} dates x {mmss_panel.shape[1]} tenors')
print(f'Tenors: {mmss_panel.columns.tolist()}')
print(f'Range: {mmss_panel.index.min()} to {mmss_panel.index.max()}')
mmss_panel.tail()

## Rolling PCA → Residuals → Z-scores

Uses the production `_rolling_pca_surface` engine from `irswap_pca_rv_scanner.py`.

In [ ]:
pca_cfg = IRSwapPCARVConfig(
    pca_window_days=cfg['pca_window'],
    pca_input=cfg['pca_input'],
    n_components=cfg['n_components'],
    zscore_lookback_days=cfg['zscore_window'],
)

t0 = time.time()
residuals, zscores, var_exp, loadings_dict, reconstructed = _rolling_pca_surface(
    mmss_panel, pca_cfg,
)
print(f'PCA scan: {time.time()-t0:.1f}s')
print(f'Residuals non-NaN: {residuals.notna().sum().to_dict()}')
print(f'Z-scores non-NaN:  {zscores.notna().sum().to_dict()}')

# Variance explained by top 3 PCs
ve_recent = var_exp.dropna().tail(60)
if not ve_recent.empty:
    print(f'\nVariance explained (last 60d avg): {ve_recent.mean().to_dict()}')

## Cumulative Residuals + OU Analytics

In [ ]:
# Cumulate residuals — the OU process lives on the cumulated series
cum_residuals = residuals.cumsum()

# OU diagnostics per tenor
print(f'{"Tenor":>6s}  {"OU mu":>8s}  {"OU HL":>7s}  {"OU theta":>9s}  {"OU sigma":>9s}')
print('-' * 50)
for t in cfg['tenors']:
    if t not in cum_residuals.columns:
        continue
    s = cum_residuals[t].dropna()
    if len(s) < 100:
        continue
    ou = _ou_params(s.iloc[-cfg['pca_window']:])
    print(f'{t:>6s}  {ou["mu"]:>8.2f}  {ou["half_life"]:>7.0f}d  {ou["theta"]:>9.3f}  {ou["sigma"]:>9.3f}')

In [ ]:
# Plot cumulative residuals
plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    ylabel_left='Cumulative PCA Residual (bp)', title='MMSS PCA Residuals (cumulated)',
)
for t in cfg['tenors']:
    if t in cum_residuals.columns:
        s = cum_residuals[t].loc[cfg['bt_start']:cfg['bt_end']]
        plot(s.rename(t), which='left', indicators=[
            {'kind': 'last', 'hide': True},
            {'kind': 'half_life', 'hide': True},
        ])
legend(show_date=True)
plt.show()

## Z-score Surface & Signal Generation

In [ ]:
# Plot z-score surface
z_bt = zscores.loc[cfg['bt_start']:cfg['bt_end']]

plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    ylabel_left='PCA Residual Z-score', title='MMSS PCA Dislocation Z-scores',
)
for t in cfg['tenors']:
    if t in z_bt.columns:
        plot(z_bt[t].rename(t), which='left', indicators=[{'kind': 'last', 'hide': True}])
legend(show_date=True)
plt.show()

# Current snapshot
latest_z = zscores.dropna(how='all').iloc[-1]
print('Current PCA dislocation z-scores:')
for t in cfg['tenors']:
    if t in latest_z.index and pd.notna(latest_z[t]):
        z_val = latest_z[t]
        tag = 'CHEAP (buy spread)' if z_val < -cfg['z_entry'] else ('RICH (sell spread)' if z_val > cfg['z_entry'] else 'neutral')
        print(f'  {t:>4s}: z = {z_val:+.2f}  {tag}')

In [ ]:
# Generate entry/exit signals per tenor
def generate_pca_signals(zscores, z_entry, z_exit, max_hold):
    signals = pd.DataFrame(0, index=zscores.index, columns=zscores.columns, dtype=np.int8)
    for t in zscores.columns:
        z = zscores[t].values
        sig = np.zeros(len(z), dtype=np.int8)
        pos = 0; hold = 0
        for i in range(len(z)):
            v = z[i]
            if np.isnan(v):
                sig[i] = 0; pos = 0; hold = 0; continue
            if pos == 0:
                if v < -z_entry: pos = +1; hold = 0   # cheap → buy spread (receive)
                elif v > z_entry: pos = -1; hold = 0   # rich → sell spread (pay)
            else:
                hold += 1
                exit_now = hold >= max_hold
                if not exit_now:
                    if pos == +1 and v >= -z_exit: exit_now = True
                    elif pos == -1 and v <= z_exit: exit_now = True
                if exit_now: pos = 0; hold = 0
            sig[i] = pos
        signals[t] = sig
    return signals

pca_signals = generate_pca_signals(zscores, cfg['z_entry'], cfg['z_exit'], cfg['max_hold'])

for t in cfg['tenors']:
    if t in pca_signals.columns:
        sig_bt = pca_signals[t].loc[cfg['bt_start']:cfg['bt_end']]
        entries = ((sig_bt != 0) & (sig_bt.shift(1).fillna(0) == 0)).sum()
        print(f'  {t}: {entries} trades')

## Vectorized P&L Preview

In [ ]:
h_bt = mmss_panel.loc[cfg['bt_start']:cfg['bt_end']]
preview = []
equity_curves = {}

for t in cfg['tenors']:
    if t not in pca_signals.columns or t not in h_bt.columns:
        continue
    sig = pca_signals[t].loc[cfg['bt_start']:cfg['bt_end']].reindex(h_bt.index).fillna(0)
    # Buy spread (sig=+1) profits when MMSS increases (narrows)
    daily = sig.shift(1) * h_bt[t].diff()
    daily = daily.fillna(0)
    cum = daily.cumsum()
    equity_curves[t] = cum

    sh = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
    mdd = (cum - cum.cummax()).min()
    n_trades = ((sig != 0) & (sig.shift(1).fillna(0) == 0)).sum()
    
    # OU on cum residuals
    cr = cum_residuals[t].loc[cfg['bt_start']:cfg['bt_end']].dropna()
    ou = _ou_params(cr) if len(cr) > 60 else {'half_life': np.nan}

    preview.append(dict(
        Tenor=t, Sharpe=round(sh, 3), Total_bp=round(cum.iloc[-1], 1),
        MaxDD_bp=round(mdd, 1), Trades=int(n_trades),
        OU_HL=round(ou['half_life'], 0) if np.isfinite(ou.get('half_life', np.nan)) else np.nan,
    ))

preview_df = pd.DataFrame(preview).sort_values('Sharpe', ascending=False)
preview_df

In [ ]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    ylabel_left='Cumulative P&L (bp)', title='PCA Dislocation — Per-Tenor Equity Curves',
)
for t, eq in equity_curves.items():
    plot(eq.rename(t), which='left', indicators=[{'kind': 'last', 'hide': True}])
legend(show_date=True)
plt.show()

## Curve & Fly Combinations (from PCA residual z-scores)

In [ ]:
# Generate RV packages from PCA z-score differentials
pkg_preview = []
pkg_curves = {}

# Curves: when one tenor is cheap (z<0) and another is rich (z>0), pair them
for front, back in itertools.combinations(cfg['tenors'], 2):
    if front not in pca_signals.columns or back not in pca_signals.columns:
        continue
    if front not in h_bt.columns or back not in h_bt.columns:
        continue
    
    # Z-score differential: z_back - z_front
    z_diff = zscores[back] - zscores[front]
    z_diff_bt = z_diff.loc[cfg['bt_start']:cfg['bt_end']]
    
    # Signal on the differential
    sig = np.zeros(len(z_diff_bt), dtype=np.int8)
    pos = 0; hold = 0
    for i in range(len(z_diff_bt)):
        v = z_diff_bt.iloc[i]
        if np.isnan(v): sig[i] = 0; pos = 0; hold = 0; continue
        if pos == 0:
            if v > cfg['z_entry']: pos = -1; hold = 0
            elif v < -cfg['z_entry']: pos = +1; hold = 0
        else:
            hold += 1
            exit_now = hold >= cfg['max_hold']
            if not exit_now:
                if pos == -1 and v <= cfg['z_exit']: exit_now = True
                elif pos == +1 and v >= -cfg['z_exit']: exit_now = True
            if exit_now: pos = 0; hold = 0
        sig[i] = pos
    sig_s = pd.Series(sig, index=z_diff_bt.index)
    
    # P&L: buy=receive back spread - pay front spread
    d_back = h_bt[back].diff()
    d_front = h_bt[front].diff()
    daily = sig_s.shift(1) * (d_back - d_front)
    daily = daily.fillna(0)
    cum = daily.cumsum()
    
    name = f'{front.replace("Y","")}/{back.replace("Y","")}'
    pkg_curves[name] = cum
    sh = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
    mdd = (cum - cum.cummax()).min()
    n_trades = ((sig_s != 0) & (sig_s.shift(1).fillna(0) == 0)).sum()
    pkg_preview.append(dict(Package=name, Kind='curve', Sharpe=round(sh, 3),
                            Total_bp=round(cum.iloc[-1], 1), MaxDD_bp=round(mdd, 1), Trades=int(n_trades)))

# Flies: z_belly vs avg(z_wings)
for front, belly, back in itertools.combinations(cfg['tenors'], 3):
    if any(t not in zscores.columns or t not in h_bt.columns for t in [front, belly, back]):
        continue
    z_fly = 2*zscores[belly] - zscores[front] - zscores[back]
    z_fly_bt = z_fly.loc[cfg['bt_start']:cfg['bt_end']]
    
    sig = np.zeros(len(z_fly_bt), dtype=np.int8)
    pos = 0; hold = 0
    for i in range(len(z_fly_bt)):
        v = z_fly_bt.iloc[i]
        if np.isnan(v): sig[i] = 0; pos = 0; hold = 0; continue
        if pos == 0:
            if v > cfg['z_entry']: pos = -1; hold = 0    # belly rich → sell belly
            elif v < -cfg['z_entry']: pos = +1; hold = 0  # belly cheap → buy belly
        else:
            hold += 1
            exit_now = hold >= cfg['max_hold']
            if not exit_now:
                if pos == -1 and v <= cfg['z_exit']: exit_now = True
                elif pos == +1 and v >= -cfg['z_exit']: exit_now = True
            if exit_now: pos = 0; hold = 0
        sig[i] = pos
    sig_s = pd.Series(sig, index=z_fly_bt.index)
    
    d_f, d_m, d_b = h_bt[front].diff(), h_bt[belly].diff(), h_bt[back].diff()
    daily = sig_s.shift(1) * (2*d_m - d_f - d_b)
    daily = daily.fillna(0)
    cum = daily.cumsum()
    
    name = f'{front.replace("Y","")}/{belly.replace("Y","")}/{back.replace("Y","")}'
    pkg_curves[name] = cum
    sh = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
    mdd = (cum - cum.cummax()).min()
    n_trades = ((sig_s != 0) & (sig_s.shift(1).fillna(0) == 0)).sum()
    pkg_preview.append(dict(Package=name, Kind='fly', Sharpe=round(sh, 3),
                            Total_bp=round(cum.iloc[-1], 1), MaxDD_bp=round(mdd, 1), Trades=int(n_trades)))

pkg_df = pd.DataFrame(pkg_preview).sort_values('Sharpe', ascending=False)
print(f'\n{len(pkg_df)} packages ({(pkg_df["Kind"]=="curve").sum()} curves, {(pkg_df["Kind"]=="fly").sum()} flies)')
print(f'Positive Sharpe: {(pkg_df["Sharpe"] > 0).sum()}/{len(pkg_df)}')
print(f'\nTop 20:')
pkg_df.head(20)

In [ ]:
# Plot top 10 package equity curves
top10 = pkg_df.head(10)['Package'].tolist()

plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    ylabel_left='Cumulative P&L (bp)', title='PCA Dislocation — Top 10 Curve/Fly Packages',
)
for name in top10:
    if name in pkg_curves:
        plot(pkg_curves[name].rename(name), which='left', indicators=[{'kind': 'last', 'hide': True}])
legend(show_date=True)
plt.show()

## Summary: PCA Outrights vs TFP Outrights vs TFP Packages

In [ ]:
print('PCA dislocation outrights:')
for _, row in preview_df.iterrows():
    print(f'  {row["Tenor"]:>4s}: Sharpe={row["Sharpe"]:+.3f}  Total={row["Total_bp"]:>7.1f}bp  Trades={row["Trades"]}  OU_HL={row["OU_HL"]}d')

print(f'\nPCA curve/fly packages (positive Sharpe only):')
pos = pkg_df[pkg_df['Sharpe'] > 0]
if pos.empty:
    print('  None with positive Sharpe.')
else:
    for _, row in pos.iterrows():
        print(f'  {row["Package"]:>10s} ({row["Kind"]}): Sharpe={row["Sharpe"]:+.3f}  Total={row["Total_bp"]:>7.1f}bp  Trades={row["Trades"]}')

print(f'\nAggregate stats:')
print(f'  Outright median Sharpe: {preview_df["Sharpe"].median():.3f}')
print(f'  Curves median Sharpe:   {pkg_df[pkg_df["Kind"]=="curve"]["Sharpe"].median():.3f}')
print(f'  Flies median Sharpe:    {pkg_df[pkg_df["Kind"]=="fly"]["Sharpe"].median():.3f}')